# 04: Unity Catalog Volume & Storage Lab


Key forms:

```text
Table        → catalog.schema.table
Volume file  → /Volumes/catalog/schema/volume/path
Azure URI    → abfss://container@account.dfs.core.windows.net/path
```


In [0]:
import sys
import os
from pathlib import Path

marker = "RETAIL_LAB.json"
cwd = Path(os.getcwd()).resolve()
PROJECT_ROOT = None

for candidate in [cwd, *cwd.parents]:
    if (candidate / marker).exists():
        PROJECT_ROOT = str(candidate)
        break

if PROJECT_ROOT is None:
    for search_root in [Path("/Workspace/Users"), Path("/Workspace/Shared"), Path("/Workspace")]:
        if not search_root.exists():
            continue
        for current_root, dirs, files in os.walk(search_root):
            dirs[:] = [d for d in dirs if not d.startswith(".") and d not in {"Trash", ".Trash"}]
            if marker in files:
                PROJECT_ROOT = current_root
                break
        if PROJECT_ROOT:
            break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project not found. Import the complete ZIP into Databricks Workspace."
    )

SRC_ROOT = f"{PROJECT_ROOT}/src"
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

from bootstrap import ensure_lab

ctx = ensure_lab(
    spark=spark,
    project_root=PROJECT_ROOT,
)

print("Project root :", PROJECT_ROOT)
print("Catalog      :", ctx.catalog)
print("Schema       :", ctx.schema)
print("Volume       :", ctx.volume)
print("Source root  :", ctx.source_root)


In [0]:
orders_df = (
    spark.read.option("header", True)
    .csv(f"{ctx.source_root}/orders.csv")
)

print("Orders:", orders_df.count())
display(orders_df.limit(20))


In [0]:
parquet_path = f"{ctx.output_root}/orders_parquet"

orders_df.write.mode("overwrite").parquet(parquet_path)

orders_parquet_df = spark.read.parquet(parquet_path)
print("Parquet path:", parquet_path)
assert orders_parquet_df.count() == 126


In [0]:
managed_table = ctx.table("lab_orders_managed")

(
    orders_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(managed_table)
)

print("Managed table:", managed_table)
assert spark.table(managed_table).count() == 126


In [0]:
from storage import abfss_uri

example = abfss_uri(
    container="landing",
    storage_account="mystorageaccount",
    relative_path="retail/orders",
)

print("Example ABFSS URI:")
print(example)
